# Reddit feed: BST vs. treap

Install the packages listed in treap/requirements.txt. The notebook uses synthetic chronological posts when no Reddit dump is present. Place a compatible dump at data/RC_2019-04.zst to use real posts.

## Input and sample size

In [ ]:
from pathlib import Path
import random
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CMakeLists.txt").exists())
sys.path.insert(0, str(ROOT))
DATA_PATH = ROOT / "data" / "RC_2019-04.zst"
POST_LIMIT = 2_000

# **WE CAN MODIFY THE MAX_ROWS TO LOAD POSTS FROM DATASET**

In [ ]:
from treap.dataset_utils import stream_reddit_zst

if DATA_PATH.is_file():
    posts = list(stream_reddit_zst(DATA_PATH, max_rows=POST_LIMIT))
else:
    rng = random.Random(42)
    posts = [(f"post-{i}", i, rng.randrange(1_000)) for i in range(POST_LIMIT)]
posts = list({post[0]: post for post in posts}.values())
posts.sort(key=lambda post: (post[1], post[0]))
if not posts:
    raise ValueError("No usable posts")
print(f"{len(posts):,} posts from {DATA_PATH.name if DATA_PATH.is_file() else 'the synthetic feed'}")

# *****LOAD REAL POSTS INTO TREAP BY MODIFYING SIZE*****

In [ ]:
from treap.trees import TreapFeed,validate_treap

# posts is assumed to be a list like:
# [(post_id, timestamp, score), ...]
# e.g. loaded from your real dataset

MAX_POSTS_TO_LOAD = 1_000_000   # change this to any size you want

def load_posts_into_treap(posts, max_posts=MAX_POSTS_TO_LOAD):
    feed = TreapFeed()

    # how many we will actually load (in case dataset is smaller)
    n = min(max_posts, len(posts))

    for i in range(n):
        post_id, timestamp, score = posts[i]
        feed.addPost(post_id, timestamp, score)

    print(f"Loaded {n} posts into Treap for console demo.")
    return feed

# ---------- usage ----------

feed = load_posts_into_treap(posts, max_posts=1_000_000)

# preview first 10 posts as: id, score, timestamp
print("\nFirst 10 posts (id, score, timestamp):")
for p in posts[:10]:
    post_id, timestamp, score = p
    print(post_id, score, timestamp)

# validate
validate_treap(feed)   # prints summary

# *****LOAD REAL POSTS INTO BST BY MODIFYING SIZE*****

In [ ]:
from treap.trees import BSTFeed,validate_bst

# posts is assumed to be a list like:
# [(post_id, timestamp, score), ...]
# e.g. loaded from your real dataset

MAX_POSTS_TO_LOAD = 1_000_000   # change this to any size you want

def load_posts_into_bst(posts, max_posts=MAX_POSTS_TO_LOAD):
    feed = BSTFeed()

    # how many we will actually load (in case dataset is smaller)
    n = min(max_posts, len(posts))

    for i in range(n):
        post_id, timestamp, score = posts[i]
        feed.addPost(post_id, timestamp, score)

    print(f"Loaded {n} posts into Treap for console demo.")
    return feed

# ---------- usage ----------

feed = load_posts_into_bst(posts, max_posts=10000)

# preview first 10 posts as: id, score, timestamp
print("\nFirst 10 posts (id, score, timestamp):")
for p in posts[:10]:
    post_id, timestamp, score = p
    print(post_id, score, timestamp)


# validate
validate_bst(feed)   # prints summary

# **WE CAN MODIFY THE SIZES TO LOAD POSTS FROM DATASET**

In [ ]:
import pandas as pd
from treap.experiments import run_experiments_for_sizes

sizes = sorted({min(len(posts), size) for size in (250, 500, 1_000, 2_000)})
df = pd.DataFrame(run_experiments_for_sizes(posts, sizes, n_search=30))
df

In [ ]:
df.style.format({
    "avg_insert_time": "{:.2e}",
    "avg_search_time": "{:.2e}",
    "avg_delete_time": "{:.2e}",
    "balancing_factor": "{:.2f}"
})


## 6. Visualizations (Comparison Dashboard)


In [ ]:
import matplotlib.pyplot as plt

# Height vs N
plt.figure()
for struct in ["BST", "Treap"]:
    sub = df[df["structure"] == struct]
    plt.plot(sub["n_posts"], sub["height"], marker="o", label=struct)

plt.xlabel("Number of posts (N)")
plt.ylabel("Tree height")
plt.title("Tree Height vs Number of Posts")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure()
for struct in ["BST", "Treap"]:
    sub = df[df["structure"] == struct]
    plt.plot(sub["n_posts"], sub["balancing_factor"], marker="o", label=struct)

plt.xlabel("Number of posts (N)")
plt.ylabel("Balancing factor (height / log2(N))")
plt.title("Balancing Factor vs Number of Posts")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure()
for struct in ["BST", "Treap"]:
    sub = df[df["structure"] == struct]
    plt.plot(sub["n_posts"], sub["avg_insert_time"], marker="o", label=struct)

plt.xlabel("Number of posts (N)")
plt.ylabel("Average insertion time (seconds)")
plt.title("Average Insertion Time vs N")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure()
for struct in ["BST", "Treap"]:
    sub = df[df["structure"] == struct]
    plt.plot(sub["n_posts"], sub["avg_search_time"], marker="o", label=struct)

plt.xlabel("Number of posts (N)")
plt.ylabel("Average search time (seconds)")
plt.title("Average getMostPopular() Time vs N")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure()
for struct in ["BST", "Treap"]:
    sub = df[df["structure"] == struct]
    plt.plot(sub["n_posts"], sub["avg_delete_time"], marker="o", label=struct)

plt.xlabel("Number of posts (N)")
plt.ylabel("Average deletion time (seconds)")
plt.title("Average Deletion Time vs N")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure()
sub = df[df["structure"] == "Treap"]
plt.plot(sub["n_posts"], sub["rotations"], marker="o")

plt.xlabel("Number of posts (N)")
plt.ylabel("Total rotations during insertion")
plt.title("Treap Rotations vs Number of Posts")
plt.grid(True)
plt.show()


# Bonus Implementations

## 1. Structural Visualization of BST and Treap
(Screen captures of tree visualizations)

## 2. Treap Union Operation
(Explanation + code snippet or description)

## 3. Treap Intersection Operation (Optional)
(Short explanation)


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def build_graph_from_tree(node, graph=None, pos=None, x=0, y=0, dx=1.0):
    if graph is None:
        graph = nx.DiGraph()
    if pos is None:
        pos = {}

    if node is None:
        return graph, pos

    label = f"{node.post_id}\n{node.score}"
    graph.add_node(label)
    pos[label] = (x, y)

    if node.left:
        left_label = f"{node.left.post_id}\n{node.left.score}"
        graph.add_edge(label, left_label)
        build_graph_from_tree(node.left, graph, pos, x - dx, y - 1, dx / 2)

    if node.right:
        right_label = f"{node.right.post_id}\n{node.right.score}"
        graph.add_edge(label, right_label)
        build_graph_from_tree(node.right, graph, pos, x + dx, y - 1, dx / 2)

    return graph, pos


def draw_tree(root, title):
    graph, pos = build_graph_from_tree(root)

    plt.figure(figsize=(10, 6))
    nx.draw(graph, pos,
            with_labels=True,
            node_size=2000,
            node_color="lightblue",
            font_size=8)
    plt.title(title)
    plt.axis("off")
    plt.show()


In [ ]:
from treap.trees import BSTFeed, TreapFeed

visual_posts = [
    ("ejualnb", 1554076800, 55),
    ("ejualnc", 1554076800, 12),
    ("ejualnd", 1554076800, 27),
    ("ejualne", 1554076800, 14),
    ("ejualnl", 1554076809, 13),
]

bst = BSTFeed()
treap = TreapFeed()

for p in visual_posts:
    bst.addPost(*p)
    treap.addPost(*p)

draw_tree(bst.root, "BST structure")
draw_tree(treap.root, "Treap structure")

In [ ]:
from treap.trees import TreapNode, TreapFeed

def treap_union(t1, t2):
    """
    Union of two Treaps:
    - Insert nodes of smaller treap into larger treap
    """
    merged = TreapFeed()

    def traverse_and_insert(node):
        if not node:
            return
        merged.addPost(node.post_id, node.timestamp, node.score)
        traverse_and_insert(node.left)
        traverse_and_insert(node.right)

    # Insert all from treap t1
    traverse_and_insert(t1.root)

    # Insert all from treap t2
    traverse_and_insert(t2.root)

    return merged


In [ ]:
# Create two small treaps
t1 = TreapFeed()
t2 = TreapFeed()

for i in range(5):
    t1.addPost(f"a{i}", 100+i, i*10)

for i in range(5):
    t2.addPost(f"b{i}", 200+i, i*20)

merged = treap_union(t1, t2)

print("Merged most popular:", merged.getMostPopular())
print("Merged height:", merged.height())


In [ ]:
def treap_intersection(t1, t2):
    """
    Returns a treap containing only posts present in both trees.
    Comparison by post_id.
    """
    intersection = TreapFeed()

    ids_common = set(t1.id_map.keys()) & set(t2.id_map.keys())

    for pid in ids_common:
        n = t1.id_map[pid]
        intersection.addPost(n.post_id, n.timestamp, n.score)

    return intersection


In [ ]:
inter = treap_intersection(t1, t2)
print("Intersection root:", inter.getMostPopular())


# 1. Introduction & Theoretical Overview

## Problem Context

Modern social media platforms such as Reddit must maintain feeds that update in real time as new posts arrive and user interactions continuously change post scores. A ranking feed must balance two competing requirements:

- **Recency** – newly created posts should appear quickly.
- **Popularity** – highly upvoted posts should gain visibility.

A naïve solution based on independent data structures for time and score increases maintenance cost and leads to expensive searches and updates as datasets grow to millions of entries.

---

## BST vs Treap

A **Binary Search Tree (BST)** stores posts ordered by a single key — in this project, the post **timestamp**. Standard BSTs do not guarantee balancing. If records arrive in nearly sorted timestamp order (as real streaming data does), the tree height can grow close to `N`, making insertion, search, and deletion degrade from `O(log N)` to `O(N)`.

A **Treap** combines the properties of:

- A **Binary Search Tree** ordered by a key (timestamp)
- A **Heap** ordered by a priority (post score)

The timestamp maintains in-order traversal, while the heap property ensures that higher-scoring nodes move upward. Whenever the heap property is violated, **local rotations** rebalance the structure. This approach results in an expected **logarithmic height**, preventing degeneration even under ordered input.

---

## Project Goal

The objective of this project is to implement both structures from scratch and integrate them into a simplified Reddit-like feed manager. Their performance is evaluated using real Reddit data with the following metrics:

- **Insertion time**
- **Search time** for retrieving the most popular post
- **Deletion time**
- **Tree height / balance behaviour**
- **Rotation frequency (Treap only)**

The comparison highlights how structural balancing directly impacts real-time feed performance.


# 2. Implementation Details

---

## Data Structures

Both the **BST** and **Treap** use the same node structure containing:

- `post_id`
- `timestamp`
- `score`
- `left` and `right` pointers

### BST

Nodes are ordered by the composite key:

`(timestamp, post_id)`

This ordering ensures unique placement even when multiple posts share identical timestamps.

---

### Treap

The Treap also uses `(timestamp, post_id)` as the BST key, but treats the post **score as heap priority** using a **max-heap** rule. Higher-scoring posts rise toward the root through **rotations** whenever the heap property is violated.

---

## Fast Access Map

Both structures maintain an auxiliary dictionary:

```python
id_map: { post_id -> tree_node }



# 3. Dataset Handling and Streaming Strategy

The original Reddit dataset is approximately **16 GB compressed** and exceeds **180 GB when extracted**, making full decompression infeasible on Kaggle or typical student hardware.

Instead, the system uses a **streaming approach**:

- The `.zst` compressed file is processed line-by-line using the **zstandard** library.
- Each record is decoded on demand.
- Only the required fields are extracted:

  - `id` → `post_id`
  - `created_utc` → `timestamp`
  - `score`

---

## Streaming Generator

A custom generator function `stream_reddit_zst()` yields entries sequentially without loading the entire dataset into memory.

This generator supplies data directly to both the BST and the Treap during construction, ensuring that posts are handled only once and never stored as a full dataset.

---

## Controlled Sampling

To remain within Kaggle’s memory and runtime limits, the number of processed rows is restricted using a configurable parameter:




This sampling strategy enables consistent benchmarking while maintaining low memory consumption and stable execution times.

---

## Scalability Note

Although multi-partition Treaps (**n-Treaps**) were not required for the tested dataset sizes, the same streaming architecture can be extended to support parallel construction of multiple trees for much larger-scale experiments.



This keeps memory usage low and enables repeatable benchmarking under Kaggle’s constraints.

---

## Scalability Note

Partitioning into multiple parallel Treaps (**n-Treaps**) was unnecessary for the selected dataset sizes. However, the same streaming logic could easily scale to process millions of posts using partitioned or distributed trees.


# 4. Experimental Setup and Metrics

---

## Experimental Configuration

Experiments were conducted for dataset sizes:


For each value of N:

1. All posts were inserted into a BST and a Treap separately.
2. `getMostPopular()` was executed **200 times** to measure average search time.
3. **10% of posts** were randomly deleted to benchmark deletion performance.
4. Structural statistics were captured after all operations.

---

## Measured Metrics

- **Average insertion time**
- **Average search time** using `getMostPopular()`
- **Average deletion time**
- **Tree height**
- **Balancing factor**



# 5. Results and Visualizations

---

### Figure 1 – Tree Height vs Dataset Size

BST height increases nearly linearly with N, reaching values close to N at large sizes.  
The Treap remains near logarithmic height (20–30 levels even at N = 10,000), showing strong self-balancing behaviour.

---

### Figure 2 – Balancing Factor

BST balancing factor rises dramatically with dataset size, indicating severe imbalance.  
The Treap maintains a stable factor near **2–2.5**, confirming near-optimal balance for all tested N.

---

### Figure 3 – `getMostPopular()` Search Time

The BST must scan the entire tree to locate the maximum-score post, causing search time to grow with N.  
The Treap returns the root node instantly, keeping search time effectively constant regardless of dataset size.

---

### Figure 4 – Rotation Count in Treap

Total rotations increase with dataset size (exceeding 10,000 operations at N = 10,000).  
These rotations represent the cost of maintaining balance, which directly enables the Treap’s superior performance across all query operations.

---

### Figure 5 – Insertion Time

BST insertion time worsens as tree height increases.  
Treap insertion includes rotation overhead but remains relatively stable overall due to maintained balance.

---

### Figure 6 – Deletion Time

BST deletions degrade under poor balancing, especially when nodes lie deep in the tree.  
Treap deletions remain efficient because node rotations ensure that removals occur near balanced paths.


# 6. Challenges and Limitations

---

## Challenges

- Correctly implementing **Treap deletion logic**, which requires sequential rotations to safely remove nodes.
- Streaming the extremely large compressed Reddit dataset while minimizing memory use.
- Operating within **Kaggle’s time and RAM constraints**, which required limiting dataset sizes.

---

## Limitations

- Experiments were capped at **10,000 posts**, far smaller than real production systems which handle millions of posts.
- Advanced balanced BST variants like **AVL** or **Red-Black trees** were not tested.
- Score updates in the Treap required reinsertion, introducing churn not present in more complex dynamic heap designs.


# 7. Conclusion

This project compared a standard **Binary Search Tree (BST)** and a **Treap** for managing real-time social-media feed data derived from Reddit.

Under nearly sorted timestamp input — representative of live streams — the BST rapidly degenerated into a near-linear chain. Its height approached N, leading to poor insertion, search, and deletion performance.

The Treap, by contrast, used heap-based balancing on post scores with local rotations to maintain near-logarithmic height. This enabled stable insertion times, constant-time retrieval of the most popular post, and efficient deletions.

**Key Outcome:**

For workloads requiring continuous updates and fast popularity queries, the **Treap significantly outperforms a naïve BST and is the superior choice for feed-ranking systems.**

Future extensions may evaluate more advanced balanced trees (AVL, Red-Black trees) or hybrid ranking structures combining recency and popularity directly into composite search keys.


In [ ]:
from treap.trees import BSTFeed, TreapFeed
import random
import time

# 🔹 Choose a moderate N so it runs fast but is still meaningful
N_LIKE = min(len(posts), 2000)
N_LIKES = 2000  # number of likePost operations to benchmark

sample_posts = posts[:N_LIKE]

# Build BST and Treap with the same data
bst_like = BSTFeed()
treap_like = TreapFeed()

for pid, ts, score in sample_posts:
    bst_like.addPost(pid, ts, score)
    treap_like.addPost(pid, ts, score)

# Collect all post_ids to randomly pick from
post_ids = [pid for pid, _, _ in sample_posts]


def benchmark_like(feed, name):
    start = time.perf_counter()
    for _ in range(N_LIKES):
        pid = random.choice(post_ids)
        feed.likePost(pid)
    end = time.perf_counter()
    avg_like_time = (end - start) / N_LIKES
    print(
        f"{name}: total likePost time = {end - start:.6f} s, "
        f"avg per likePost = {avg_like_time*1e6:.2f} µs"
    )
    return avg_like_time


print(f"Benchmarking likePost with N = {N_LIKE} posts and {N_LIKES} likes...\n")
bst_like_time = benchmark_like(bst_like, "BST")
treap_like_time = benchmark_like(treap_like, "Treap")

In [ ]:
K = 5  # how many most recent posts to show

print(f"Top {K} most recent posts from BST:")
recent_bst = bst_like.getMostRecent(K)
for pid, ts, score in recent_bst:
    print(f"post_id={pid}, timestamp={ts}, score={score}")

print("\nTop {K} most recent posts from Treap:")
recent_treap = treap_like.getMostRecent(K)
for pid, ts, score in recent_treap:
    print(f"post_id={pid}, timestamp={ts}, score={score}")


In [ ]:
import numpy as np

operations = ["avg_insert_time", "avg_search_time", "avg_delete_time"]
structures = df["structure"].unique()

results = []

for struct in structures:
    sub = df[df["structure"] == struct].sort_values("n_posts")
    n = sub["n_posts"].values.astype(float)
    log_n = np.log(n)
    
    for op in operations:
        t = sub[op].values.astype(float)
        log_t = np.log(t)
        
        # Linear fit log(time) = a + b * log(N)
        b, a = np.polyfit(log_n, log_t, 1)
        
        results.append({
            "structure": struct,
            "operation": op,
            "slope_b": b
        })

results_df = pd.DataFrame(results)
results_df


In [ ]:
def interpret_slope(b):
    b_abs = abs(b)
    if b_abs < 0.15:
        return "≈ O(1)"
    elif b_abs < 0.5:
        return "≈ sublinear (close to O(log N))"
    elif 0.5 <= b_abs < 1.5:
        return "≈ O(N^%.2f)" % b
    else:
        return "superlinear (N^%.2f)" % b

results_df["empirical_complexity"] = results_df["slope_b"].apply(interpret_slope)
results_df


In [ ]:
theory_map = {
    ("BST", "avg_insert_time"): "O(h) (≈ O(N) worst, O(log N) avg)",
    ("BST", "avg_search_time"): "O(N) (getMostPopular)",
    ("BST", "avg_delete_time"): "O(h) (≈ O(N) worst, O(log N) avg)",
    ("Treap", "avg_insert_time"): "O(log N) expected",
    ("Treap", "avg_search_time"): "O(1) (getMostPopular)",
    ("Treap", "avg_delete_time"): "O(log N) expected",
}

results_df["theoretical"] = results_df.apply(
    lambda row: theory_map.get((row["structure"], row["operation"]), ""),
    axis=1
)

results_df


In [ ]:
print(f"Benchmark input: {len(posts):,} posts")

In [ ]:
# ================================
# Console-Based Control for Treap / BST (Final Version)
# ================================

from treap.trees import BSTFeed, TreapFeed


def _read_int(prompt, allow_empty=False, default=None):
    """
    Safely read an integer from input(). Re-prompts on invalid input.
    If allow_empty=True and user just presses Enter, returns default.
    """
    while True:
        s = input(prompt).strip()
        if allow_empty and s == "":
            return default
        try:
            return int(s)
        except ValueError:
            print("Please enter a valid integer.")


def run_feed_console(feed, name="Treap", initial_ids=None):
    """
    Simple console menu for interacting with a feed.

    The feed object must implement:
      - addPost(post_id, timestamp, score)
      - likePost(post_id)
      - deletePost(post_id)
      - getMostPopular() -> (post_id, ts, score) or None
      - getMostRecent(k) -> list[(post_id, ts, score)]

    We keep our own set of known_ids so we don't depend on internal id_map names.
    """
    # track post IDs we know about
    known_ids = set(initial_ids) if initial_ids is not None else set()

    print(f"\n=== {name} Feed Console ===")
    print("You can now manually test addPost, likePost, deletePost, etc.")
    print("Type the option number and press Enter.")

    while True:
        print("\n----------------------------")
        print(f"Current structure: {name}")
        print("1) Add post")
        print("2) Like post")
        print("3) Delete post")
        print("4) Get most popular post")
        print("5) Get K most recent posts")
        print("6) Show all post IDs in feed")
        print("0) Exit console")
        choice = input("Your choice: ").strip()

        # ===== Exit =====
        if choice == "0":
            print("Exiting console...")
            break

        # ===== Add post =====
        elif choice == "1":
            post_id = input("Enter post_id (string): ").strip()
            timestamp = _read_int("Enter timestamp (integer, e.g. 1, 2, 3, or Unix time): ")
            score = _read_int("Enter initial score (integer): ")

            try:
                feed.addPost(post_id, timestamp, score)
                known_ids.add(post_id)
                print(f"✅ Post added: id={post_id}, ts={timestamp}, score={score}")
            except Exception as e:
                print("❌ Error during addPost:", e)

        # ===== Like post =====
        elif choice == "2":
            post_id = input("Enter post_id to like: ").strip()

            if post_id not in known_ids:
                print(f"⚠️ Cannot like post: {post_id} — post does not exist (in console view).")
            else:
                try:
                    feed.likePost(post_id)
                    print(f"👍 Liked post: {post_id}")
                except Exception as e:
                    print("❌ Error during likePost:", e)

        # ===== Delete post =====
        elif choice == "3":
            post_id = input("Enter post_id to delete: ").strip()

            if post_id not in known_ids:
                print(f"⚠️ Cannot delete post: {post_id} — post does not exist (in console view).")
            else:
                try:
                    feed.deletePost(post_id)
                    known_ids.discard(post_id)
                    print(f"🗑️ Deleted post: {post_id}")
                except Exception as e:
                    print("❌ Error during deletePost:", e)

        # ===== Get most popular =====
        elif choice == "4":
            try:
                result = feed.getMostPopular()
                if result is None:
                    print("There are no posts in the feed.")
                else:
                    pid, ts, score = result
                    print(f"⭐ Most popular post -> id={pid}, ts={ts}, score={score}")
            except Exception as e:
                print("❌ Error during getMostPopular:", e)

        # ===== Get K most recent =====
        elif choice == "5":
            k = _read_int("Enter K (how many recent posts to show): ")
            try:
                result = feed.getMostRecent(k)
                if not result:
                    print("There are no posts in the feed.")
                else:
                    print(f"🕒 Top {k} most recent posts:")
                    for pid, ts, score in result:
                        print(f"  id={pid}, ts={ts}, score={score}")
            except Exception as e:
                print("❌ Error during getMostRecent:", e)

        # ===== Show all IDs (from known_ids) =====
        elif choice == "6":
            if not known_ids:
                print("There are no posts tracked by the console yet.")
            else:
                print("Current post IDs in the feed (console tracking):")
                for pid in known_ids:
                    print(" ", pid)

        else:
            print("Invalid choice, please try again.")
